In [1]:
var('n e_t u_bar v_bar f_4t f_5t g_4t g_5t Xi dt du dv gamma F_phi e_phi y K G m')
var('u_bar m G eta gamma e_t n Xi')
var('y y_0 h TE y_1PN y_15PN y_2PN y_25PN Xi chi_A chi_S delta omega')
var('e_t gamma y_inv e0 omega_dash A B flagTail e0 x0 flagHF SO chi w0 f0 j f_var f0_var e y0 kappaA kappaS horiz z')

(e_t,
 gamma,
 y_inv,
 e0,
 omega_dash,
 A,
 B,
 flagTail,
 e0,
 x0,
 flagHF,
 SO,
 chi,
 w0,
 f0,
 j,
 f_var,
 f0_var,
 e,
 y0,
 kappaA,
 kappaS,
 horiz,
 z)

In [2]:
import re
filename = "henry/de_dt_final.txt"

with open(filename, "r") as f:
    content = f.read()


match = re.search(r"de_dt_final\s*=\s*(.*)", content, re.DOTALL)
if match:
    expr_str = match.group(1).strip()
    de_dt = ((sage_eval(expr_str, locals=globals())).subs({e:e_t})).expand()
    #*(1/(Xi^3*G))



filename = "henry/dx_dt_final.txt"

with open(filename, "r") as f:
    content = f.read()


match = re.search(r"dx_dt_final\s*=\s*(.*)", content, re.DOTALL)
if match:
    expr_str = match.group(1).strip()
    dx_dt = ((sage_eval(expr_str, locals=globals())).subs({e:e_t})).expand()
    #*(Xi^3/G)


filename = "henry/dx_dt_horiz.txt"

with open(filename, "r") as f:
    content = f.read()


match = re.search(r"dx_dt_horiz\s*=\s*(.*)", content, re.DOTALL)
if match:
    expr_str = match.group(1).strip()
    dx_dt_horiz = (((sage_eval(expr_str, locals=globals())).subs({e:e_t})).expand())*flagHF   


filename = "henry/de_dt_horiz.txt"

with open(filename, "r") as f:
    content = f.read()


match = re.search(r"de_dt_horiz\s*=\s*(.*)", content, re.DOTALL)
if match:
    expr_str = match.group(1).strip()
    de_dt_horiz = (((sage_eval(expr_str, locals=globals())).subs({e:e_t})).expand())*flagHF    

In [3]:
filename = "henry/de_nonspinning_final.txt"

with open(filename, "r") as f:
    content = f.read()


match = re.search(r"de_nonspinning_final\s*=\s*(.*)", content, re.DOTALL)
if match:
    expr_str = match.group(1).strip()
    de_nonspinning_dt = (sage_eval(expr_str, locals=globals())).expand()

filename = "henry/dx_nonspinning_final.txt"

with open(filename, "r") as f:
    content = f.read()


match = re.search(r"dx_nonspinning_final\s*=\s*(.*)", content, re.DOTALL)
if match:
    expr_str = match.group(1).strip()
    dx_nonspinning_dt = (sage_eval(expr_str, locals=globals())).expand()

In [4]:
import re
filename = "henry/xe.txt"

with open(filename, "r") as f:
    content = f.read()


match = re.search(r"xe_final\s*=\s*(.*)", content, re.DOTALL)
if match:
    expr_str = match.group(1).strip()
    xe_expr = ((sage_eval(expr_str, locals=globals()))*(e^(12/19)/e0^(12/19))).subs({e:e_t}).expand()

In [5]:
def truncate_var(expr,var, order=2):
    """
    Truncate a Sage expression in Xi to the given order (default 3).
    Keeps terms up to Xi^order and discards higher powers.
    """
    truncated = sum(expr.coefficient(var,n) * var^n for n in range(order + 1))
    return truncated

**Factor PN expressions to reduce computational cost**

**Factor the dx_dt expressions and include non spinning terms from Michael, Faye,...**

In [6]:
dx_dt_N_term=(dx_dt.coefficient(Xi,0).subs({(-e_t^2+1)^(-7/2):y}).factor().subs({y:(-e_t^2+1)^(-7/2)}))

In [7]:
dx_dt_1PN_term_base=(dx_dt.coefficient(Xi,2).coefficient(x^6).subs({(-e_t^2+1)^(-9/2):y}).factor().subs({y:(-e_t^2+1)^(-9/2)}))
dx_dt_1PN_term_num=(dx_dt_1PN_term_base.numerator().collect(e_t))*x^6
dx_dt_1PN_term_den=dx_dt_1PN_term_base.denominator()
dx_dt_1PN_term=dx_dt_1PN_term_num/dx_dt_1PN_term_den

In [8]:
dx_dt_15PN_term_base=dx_dt.coefficient(Xi,3).coefficient(x^(13/2)).subs({e_t^10 - 5*e_t^8 + 10*e_t^6 - 10*e_t^4 + 5*e_t^2 - 1:-(-e_t^2+1)^5})
dx_dt_15PN_term_num=(dx_dt_15PN_term_base.numerator().collect(e_t))*x^(13/2)
dx_dt_15PN_term_den=dx_dt_15PN_term_base.denominator()
dx_dt_15PN_term=dx_dt_15PN_term_num/dx_dt_15PN_term_den

In [9]:
show((dx_dt.coefficient(Xi,3)-dx_dt_15PN_term).subs({-e_t^2+1:y}).expand())

0

In [10]:
dx_dt_2PN_term_base_0=(dx_nonspinning_dt.coefficient(Xi,4)+dx_dt.coefficient(Xi,4).coefficient(SO,1)*SO+dx_dt.coefficient(Xi,4).coefficient(SO,2)*SO^2)
dx_dt_2PN_term_base=(dx_dt_2PN_term_base_0.coefficient(x^7).subs({(-e_t^2+1):y}).subs({(e_t^2-1):-y}))
dx_dt_2PN_term_pt_1=(dx_dt_2PN_term_base.coefficient(y^(-5)).numerator().collect(e_t)/dx_dt_2PN_term_base.coefficient(y^(-5)).denominator())*(1/y^5)
dx_dt_2PN_term_pt_2=(dx_dt_2PN_term_base.coefficient(y^(-11/2)).numerator().collect(e_t)/dx_dt_2PN_term_base.coefficient(y^(-11/2)).denominator())*(1/y^(11/2))
dx_dt_2PN_term=((dx_dt_2PN_term_pt_1+dx_dt_2PN_term_pt_2).subs({y:-e_t^2+1}))*x^7

In [11]:
show((dx_dt_2PN_term.subs({-e_t^2+1:y}).expand()-dx_dt_2PN_term.subs({-e_t^2+1:y}).expand().coefficient(SO,0)-dx_dt.coefficient(Xi,4).subs({-e_t^2+1:y})+dx_dt.coefficient(Xi,4).subs({-e_t^2+1:y}).coefficient(SO,0)).expand())

0

In [12]:
show((dx_dt_2PN_term.subs({-e_t^2+1:y}).expand().coefficient(SO,0)-dx_nonspinning_dt.coefficient(Xi,4).subs({-e_t^2+1:y})).expand())

0

In [13]:
dx_dt_25PN_term_base_0=(dx_dt.coefficient(Xi,5)+dx_dt_horiz.coefficient(Xi,5)).coefficient(x^(15/2)).subs({(-e_t^2+1)^(1/2):y^(1/2)}).subs({e_t^12 - 6*e_t^10 + 15*e_t^8 - 20*e_t^6 + 15*e_t^4 - 6*e_t^2 + 1:y^6})
dx_dt_25PN_term_pt_1=(dx_dt_25PN_term_base_0.coefficient(y^(-6)).numerator().collect(e_t)/dx_dt_25PN_term_base_0.coefficient(y^(-6)).denominator())*(1/y^6)
dx_dt_25PN_term_pt_2=(dx_dt_25PN_term_base_0.coefficient(y^(-11/2)).numerator().collect(e_t)/dx_dt_25PN_term_base_0.coefficient(y^(-11/2)).denominator())*(1/y^(11/2))

In [14]:
dx_dt_25PN_term_base_1=(dx_nonspinning_dt.coefficient(Xi,5).coefficient(x^(15/2)).subs({-e_t^2+1:y}).subs({e_t^2-1:-y}))
dx_dt_25PN_term_pt_3=(dx_dt_25PN_term_base_1.subs({y:1/z}).coefficient(z,0).numerator().collect(e_t)/dx_dt_25PN_term_base_1.subs({y:1/z}).coefficient(z,0).denominator())
dx_dt_25PN_term_pt_4=(dx_dt_25PN_term_base_1.coefficient(y^(-1)).numerator().collect(e_t)/dx_dt_25PN_term_base_1.coefficient(y^(-1)).denominator())*(1/y)
dx_dt_25PN_term_pt_5=(dx_dt_25PN_term_base_1.coefficient(y^(-3/2)).numerator().collect(e_t)/dx_dt_25PN_term_base_1.coefficient(y^(-3/2)).denominator())*(1/y^(3/2))
dx_dt_25PN_term=((dx_dt_25PN_term_pt_1+dx_dt_25PN_term_pt_2+dx_dt_25PN_term_pt_3+dx_dt_25PN_term_pt_4+dx_dt_25PN_term_pt_5).subs({y:-e_t^2+1}))*x^(15/2)

**Verify 2.5 PN factorisation (all good)**

In [15]:
hf_25=dx_dt_25PN_term.subs({e_t^2-1:-y}).subs({-e_t^2+1:y}).expand().coefficient(flagHF)
hf_25_og=dx_dt_horiz.coefficient(Xi,5).coefficient(flagHF).subs({e_t^12 - 6*e_t^10 + 15*e_t^8 - 20*e_t^6 + 15*e_t^4 - 6*e_t^2 + 1:y^6}).subs({e_t^2-1:-y}).subs({-e_t^2+1:y}).expand()
show((hf_25_og-hf_25).expand())

0

In [16]:
ns_25=dx_dt_25PN_term.subs({e_t^2-1:-y}).subs({-e_t^2+1:y}).expand().coefficient(SO,0).coefficient(flagHF,0)
ns_25_og=dx_nonspinning_dt.coefficient(Xi,5).subs({e_t^2-1:-y}).subs({-e_t^2+1:y}).expand()

In [17]:
show((ns_25-ns_25_og).expand())

0

In [18]:
s_25=dx_dt_25PN_term.subs({e_t^2-1:-y}).subs({-e_t^2+1:y}).expand().coefficient(SO)
s_25_og=dx_dt.coefficient(Xi,5).coefficient(SO).subs({e_t^12 - 6*e_t^10 + 15*e_t^8 - 20*e_t^6 + 15*e_t^4 - 6*e_t^2 + 1:y^6}).subs({e_t^2-1:-y}).subs({-e_t^2+1:y}).expand()

In [19]:
show((s_25-s_25_og).expand())

0

In [20]:
dx_dt_3PN_term_base_0=((dx_dt.coefficient(Xi,6)+dx_nonspinning_dt.coefficient(Xi,6)).expand().coefficient(x^8).subs({e_t^12 - 6*e_t^10 + 15*e_t^8 - 20*e_t^6 + 15*e_t^4 - 6*e_t^2 + 1:y^6}).subs({(-e_t^2+1):y}))
dx_dt_3PN_term_pt_1=(dx_dt_3PN_term_base_0.coefficient(y^(-6)).numerator().collect(e_t)/dx_dt_3PN_term_base_0.coefficient(y^(-6)).denominator())*(1/y^6)
dx_dt_3PN_term_pt_2=(dx_dt_3PN_term_base_0.coefficient(y^(-13/2)).numerator().collect(e_t)/dx_dt_3PN_term_base_0.coefficient(y^(-13/2)).denominator())*(1/y^(13/2))
dx_dt_3PN_term_pt_3=(dx_dt_3PN_term_base_0.subs({y:1/z}).coefficient(z,0).numerator().collect(e_t)/dx_dt_3PN_term_base_0.subs({y:1/z}).coefficient(z,0).denominator())
dx_dt_3PN_term=((dx_dt_3PN_term_pt_1+dx_dt_3PN_term_pt_2+dx_dt_3PN_term_pt_3).subs({y:-e_t^2+1}))*x^8

In [21]:
ns_3=dx_dt_3PN_term.expand().coefficient(SO,0).subs({e_t^12 - 6*e_t^10 + 15*e_t^8 - 20*e_t^6 + 15*e_t^4 - 6*e_t^2 + 1:y^6})
ns_3_og=dx_nonspinning_dt.coefficient(Xi,6).expand().subs({e_t^12 - 6*e_t^10 + 15*e_t^8 - 20*e_t^6 + 15*e_t^4 - 6*e_t^2 + 1:y^6})

In [22]:
show((ns_3-ns_3_og).expand())

0

In [23]:
s_3=dx_dt_3PN_term.expand().subs({e_t^12 - 6*e_t^10 + 15*e_t^8 - 20*e_t^6 + 15*e_t^4 - 6*e_t^2 + 1:y^6})-ns_3
s_3_og=dx_dt.coefficient(Xi,6).expand().subs({e_t^12 - 6*e_t^10 + 15*e_t^8 - 20*e_t^6 + 15*e_t^4 - 6*e_t^2 + 1:y^6})

In [24]:
show((s_3-s_3_og).expand())

0

In [25]:
dx_dt_final=dx_dt_N_term+dx_dt_1PN_term*Xi^2+dx_dt_15PN_term*Xi^3+dx_dt_2PN_term*Xi^4+dx_dt_25PN_term*Xi^5+dx_dt_3PN_term*Xi^6

**Factor the de_dt expressions and include non spinning terms from Michael, Faye,...**

In [26]:
de_dt_N_term=(de_dt.coefficient(Xi,0).subs({(-e_t^2+1)^(-5/2):y}).factor().subs({y:(-e_t^2+1)^(-5/2)}))

In [27]:
show((de_dt.coefficient(Xi,0)-de_dt_N_term).subs({-e_t^2+1:y}).expand())

0

In [28]:
de_dt_1PN_term_base=(de_dt.coefficient(Xi,2).coefficient(x^5).subs({(-e_t^2+1)^(-7/2):y}).factor().subs({y:(-e_t^2+1)^(-7/2)}))
de_dt_1PN_term_num=(de_dt_1PN_term_base.numerator().collect(e_t))*x^5
de_dt_1PN_term_den=de_dt_1PN_term_base.denominator()
de_dt_1PN_term=de_dt_1PN_term_num/de_dt_1PN_term_den

In [29]:
show((de_dt.coefficient(Xi,2)-de_dt_1PN_term).subs({-e_t^2+1:y}).expand())

0

In [30]:
de_dt_15PN_term_base=de_dt.coefficient(Xi,3).coefficient(x^(11/2)).subs({e_t^8 - 4*e_t^6 + 6*e_t^4 - 4*e_t^2 + 1:(-e_t^2+1)^4})
de_dt_15PN_term_num=(de_dt_15PN_term_base.numerator().collect(e_t))*x^(11/2)
de_dt_15PN_term_den=de_dt_15PN_term_base.denominator()
de_dt_15PN_term=de_dt_15PN_term_num/de_dt_15PN_term_den

In [31]:
show(de_dt.coefficient(Xi,3).subs({e_t^8 - 4*e_t^6 + 6*e_t^4 - 4*e_t^2 + 1:y^4})-de_dt_15PN_term.subs({e_t^2-1:-y}).subs({-e_t^2+1:y}).expand())

0

In [32]:
de_dt_2PN_term_base=de_dt.coefficient(Xi,4).coefficient(x^6).subs({e_t^8 - 4*e_t^6 + 6*e_t^4 - 4*e_t^2 + 1:y^4}).subs({(-e_t^2+1):y})
de_dt_2PN_term_pt_1=(de_dt_2PN_term_base.coefficient(y^(-9/2)).numerator().collect(e_t)/de_dt_2PN_term_base.coefficient(y^(-9/2)).denominator())*(1/y^(9/2))
de_dt_2PN_term_pt_2=(de_dt_2PN_term_base.coefficient(y^(-4)).numerator().collect(e_t)/de_dt_2PN_term_base.coefficient(y^(-4)).denominator())*(1/y^(4))
de_dt_2PN_term=((de_dt_2PN_term_pt_1+de_dt_2PN_term_pt_2).subs({y:-e_t^2+1}))*x^6

In [33]:
show((de_dt_2PN_term.subs({-e_t^2+1:y}).expand()-de_dt_2PN_term.subs({-e_t^2+1:y}).expand().coefficient(SO,0)-de_dt.coefficient(Xi,4).subs({-e_t^2+1:y})+de_dt.coefficient(Xi,4).subs({-e_t^2+1:y}).coefficient(SO,0)).expand())

0

In [34]:
show((de_dt_2PN_term.subs({e_t^8 - 4*e_t^6 + 6*e_t^4 - 4*e_t^2 + 1:y^4}).subs({-e_t^2+1:y}).subs({e_t^2-1:y}).expand().coefficient(SO,0)-de_dt.coefficient(Xi,4).coefficient(SO,0).subs({e_t^8 - 4*e_t^6 + 6*e_t^4 - 4*e_t^2 + 1:y^4}).subs({e_t^2-1:y}).subs({-e_t^2+1:y})).expand())

0

In [35]:
de_dt_25PN_term_base_0=(de_dt.coefficient(Xi,5)+de_dt_horiz.coefficient(Xi,5)).coefficient(x^(13/2)).subs({e_t^10 - 5*e_t^8 + 10*e_t^6 - 10*e_t^4 + 5*e_t^2 - 1:-y^5}).subs({(-e_t^2+1):y})
de_dt_25PN_term_pt_1=(de_dt_25PN_term_base_0.coefficient(y^(-5)).numerator().collect(e_t)/de_dt_25PN_term_base_0.coefficient(y^(-5)).denominator())*(1/y^5)
de_dt_25PN_term_pt_2=(de_dt_25PN_term_base_0.coefficient(y^(-9/2)).numerator().collect(e_t)/de_dt_25PN_term_base_0.coefficient(y^(-9/2)).denominator())*(1/y^(9/2))

In [36]:
show(de_dt_25PN_term_base_0)

-4267/30*SO*chi_A*delta*e_t^7*eta^2/(m*y^5) - 752/45*SO*chi_S*e_t^7*eta^3/(m*y^5) + 345811/2240*SO*chi_A*delta*e_t^7*eta/(m*y^5) - 221909/1680*SO*chi_S*e_t^7*eta^2/(m*y^5) - 1252/45*SO*chi_A*delta*e_t^5*eta^2/(m*y^(9/2)) + 626/45*SO*chi_S*e_t^5*eta^3/(m*y^(9/2)) + 345811/2240*SO*chi_S*e_t^7*eta/(m*y^5) - 1838173/540*SO*chi_A*delta*e_t^5*eta^2/(m*y^5) + 112736/135*SO*chi_S*e_t^5*eta^3/(m*y^5) + 1252/15*SO*chi_A*delta*e_t^5*eta/(m*y^(9/2)) - 5008/45*SO*chi_S*e_t^5*eta^2/(m*y^(9/2)) + 97535/24*SO*chi_A*delta*e_t^5*eta/(m*y^5) - 4593796/945*SO*chi_S*e_t^5*eta^2/(m*y^5) + 1252/15*SO*chi_S*e_t^5*eta/(m*y^(9/2)) - 9236/15*SO*chi_A*delta*e_t^3*eta^2/(m*y^(9/2)) + 4618/15*SO*chi_S*e_t^3*eta^3/(m*y^(9/2)) - 11/10*(3*chi_A^3*delta*e_t^5*eta^2/(m*y^5) + 9*chi_A*chi_S^2*delta*e_t^5*eta^2/(m*y^5) - 3*chi_A^3*delta*e_t^5*eta/(m*y^5) - 9*chi_A*chi_S^2*delta*e_t^5*eta/(m*y^5) + 27*chi_A^2*chi_S*e_t^5*eta^2/(m*y^5) + 9*chi_S^3*e_t^5*eta^2/(m*y^5) - 9*chi_A^2*chi_S*e_t^5*eta/(m*y^5) - 3*chi_S^3*e_t^5*eta/(m*y^5) + 36*chi_A^3*delta*e_t^3*eta^2/(m*y^5) + 108*chi_A*chi_S^2*delta*e_t^3*eta^2/(m*y^5) + chi_A*delta*e_t^5*eta^2/(m*y^5) - 36*chi_A^3*delta*e_t^3*eta/(m*y^5) - 108*chi_A*chi_S^2*delta*e_t^3*eta/(m*y^5) - chi_A*delta*e_t^5*eta/(m*y^5) + 324*chi_A^2*chi_S*e_t^3*eta^2/(m*y^5) + 108*chi_S^3*e_t^3*eta^2/(m*y^5) + 3*chi_S*e_t^5*eta^2/(m*y^5) - 108*chi_A^2*chi_S*e_t^3*eta/(m*y^5) - 36*chi_S^3*e_t^3*eta/(m*y^5) - chi_S*e_t^5*eta/(m*y^5) + 24*chi_A^3*delta*e_t*eta^2/(m*y^5) + 72*chi_A*chi_S^2*delta*e_t*eta^2/(m*y^5) + 12*chi_A*delta*e_t^3*eta^2/(m*y^5) - 24*chi_A^3*delta*e_t*eta/(m*y^5) - 72*chi_A*chi_S^2*delta*e_t*eta/(m*y^5) - 12*chi_A*delta*e_t^3*eta/(m*y^5) + 216*chi_A^2*chi_S*e_t*eta^2/(m*y^5) + 72*chi_S^3*e_t*eta^2/(m*y^5) + 36*chi_S*e_t^3*eta^2/(m*y^5) - 72*chi_A^2*chi_S*e_t*eta/(m*y^5) - 24*chi_S^3*e_t*eta/(m*y^5) - 12*chi_S*e_t^3*eta/(m*y^5) + 8*chi_A*delta*e_t*eta^2/(m*y^5) - 8*chi_A*delta*e_t*eta/(m*y^5) + 24*chi_S*e_t*eta^2/(m*y^5) - 8*chi_S*e_t*eta/(m*y^5))*flagHF + 97535/24*SO*chi_S*e_t^5*eta/(m*y^5) - 523579/90*SO*chi_A*delta*e_t^3*eta^2/(m*y^5) + 126833/45*SO*chi_S*e_t^3*eta^3/(m*y^5) + 9236/5*SO*chi_A*delta*e_t^3*eta/(m*y^(9/2)) - 36944/15*SO*chi_S*e_t^3*eta^2/(m*y^(9/2)) + 6885449/1260*SO*chi_A*delta*e_t^3*eta/(m*y^5) - 2152457/210*SO*chi_S*e_t^3*eta^2/(m*y^5) + 9236/5*SO*chi_S*e_t^3*eta/(m*y^(9/2)) - 2368/9*SO*chi_A*delta*e_t*eta^2/(m*y^(9/2)) + 1184/9*SO*chi_S*e_t*eta^3/(m*y^(9/2)) + 6885449/1260*SO*chi_S*e_t^3*eta/(m*y^5) - 171062/135*SO*chi_A*delta*e_t*eta^2/(m*y^5) + 105832/135*SO*chi_S*e_t*eta^3/(m*y^5) + 2368/3*SO*chi_A*delta*e_t*eta/(m*y^(9/2)) - 9472/9*SO*chi_S*e_t*eta^2/(m*y^(9/2)) + 36083/105*SO*chi_A*delta*e_t*eta/(m*y^5) - 357772/189*SO*chi_S*e_t*eta^2/(m*y^5) + 2368/3*SO*chi_S*e_t*eta/(m*y^(9/2)) + 36083/105*SO*chi_S*e_t*eta/(m*y^5)

In [37]:
de_dt_25PN_term_base_1=(de_nonspinning_dt.coefficient(Xi,5).coefficient(x^(13/2)).subs({-e_t^2+1:y}).subs({e_t^2-1:-y}))
de_dt_25PN_term_pt_3=(de_dt_25PN_term_base_1.coefficient(y^(-1/2)).numerator().collect(e_t)/de_dt_25PN_term_base_1.coefficient(y^(-1/2)).denominator())*(1/y^(1/2))
de_dt_25PN_term_pt_4=(de_dt_25PN_term_base_1.coefficient(y^(1/2)).numerator().collect(e_t)/de_dt_25PN_term_base_1.coefficient(y^(1/2)).denominator())*(y^(1/2))
de_dt_25PN_term_pt_5=(de_dt_25PN_term_base_1.coefficient(y,0).numerator().collect(e_t)/de_dt_25PN_term_base_1.coefficient(y,0).denominator())
de_dt_25PN_term=((de_dt_25PN_term_pt_1+de_dt_25PN_term_pt_2+de_dt_25PN_term_pt_3+de_dt_25PN_term_pt_4+de_dt_25PN_term_pt_5).subs({y:-e_t^2+1}))*x^(13/2)

In [38]:
e_hf_25=de_dt_25PN_term.subs({e_t^2-1:-y}).subs({-e_t^2+1:y}).expand().coefficient(flagHF)
e_hf_25_og=de_dt_horiz.coefficient(Xi,5).coefficient(flagHF).subs({e_t^10 - 5*e_t^8 + 10*e_t^6 - 10*e_t^4 + 5*e_t^2 - 1:-y^5}).subs({e_t^2-1:-y}).subs({-e_t^2+1:y}).expand()
show((e_hf_25_og-e_hf_25).expand())

0

In [39]:
e_ns_25=de_dt_25PN_term.subs({e_t^2-1:-y}).subs({-e_t^2+1:y}).expand().coefficient(SO,0).coefficient(flagHF,0)
e_ns_25_og=de_nonspinning_dt.coefficient(Xi,5).subs({e_t^2-1:-y}).subs({-e_t^2+1:y}).expand()

In [40]:
show((e_ns_25-e_ns_25_og).expand())

0

In [41]:
e_s_25=de_dt_25PN_term.subs({e_t^2-1:-y}).subs({-e_t^2+1:y}).expand().coefficient(SO)
e_s_25_og=de_dt.coefficient(Xi,5).coefficient(SO).subs({e_t^10 - 5*e_t^8 + 10*e_t^6 - 10*e_t^4 + 5*e_t^2 - 1:-y^5}).subs({e_t^2-1:-y}).subs({-e_t^2+1:y}).expand()

In [42]:
show((e_s_25-e_s_25_og).expand())

0

In [43]:
de_dt_3PN_term_base_0=((de_dt.coefficient(Xi,6)+de_nonspinning_dt.coefficient(Xi,6)).expand().coefficient(x^7).subs({e_t^10 - 5*e_t^8 + 10*e_t^6 - 10*e_t^4 + 5*e_t^2 - 1:-y^5}).subs({(-e_t^2+1):y}).subs({(e_t^2-1):-y}))
de_dt_3PN_term_base_1=(de_dt_3PN_term_base_0-(de_dt_3PN_term_base_0.coefficient((sqrt(y)+1)^(-1)))*((sqrt(y)+1)^(-1))).expand()
de_dt_3PN_term_base_2=de_dt_3PN_term_base_1+(de_dt_3PN_term_base_0.coefficient((sqrt(y)+1)^(-1)).subs({y^(-11/2):y_0}))*((sqrt(y)+1)^(-1))


In [44]:
de_dt_3PN_term_pt_1=(de_dt_3PN_term_base_2.coefficient(y^(-5)).numerator().collect(e_t)/de_dt_3PN_term_base_2.coefficient(y^(-5)).denominator())*(1/y^5)
de_dt_3PN_term_pt_2=((de_dt_3PN_term_base_2.coefficient(y^(-11/2)).numerator().collect(e_t)/de_dt_3PN_term_base_2.coefficient(y^(-11/2)).denominator())+de_dt_3PN_term_base_2.coefficient(y_0))*(1/y^(11/2))
de_dt_3PN_term_pt_3=(de_dt_3PN_term_base_2.coefficient(y^(1/2)).numerator().collect(e_t)/de_dt_3PN_term_base_2.coefficient(y^(1/2)).denominator())*(y^(1/2))


In [45]:
messy_term=(de_dt_3PN_term_base_2.coefficient(y,0)-(de_dt_3PN_term_base_2.coefficient(y_0))*(y_0)).expand()

In [46]:
de_dt_3PN_term_pt_4=(messy_term.coefficient(y,0).numerator().collect(e_t)/messy_term.coefficient(y,0).denominator())
de_dt_3PN_term=((de_dt_3PN_term_pt_1+de_dt_3PN_term_pt_2+de_dt_3PN_term_pt_3+de_dt_3PN_term_pt_4).subs({y:-e_t^2+1}))*x^7

In [47]:
e_ns_3=de_dt_3PN_term.subs({(-e_t^2+1):y}).subs({(e_t^2-1):-y}).expand().coefficient(SO,0).subs({e_t^10 - 5*e_t^8 + 10*e_t^6 - 10*e_t^4 + 5*e_t^2 - 1:-y^5})
e_ns_3_og=de_nonspinning_dt.coefficient(Xi,6).subs({(-e_t^2+1):y}).subs({(e_t^2-1):-y}).expand().subs({e_t^10 - 5*e_t^8 + 10*e_t^6 - 10*e_t^4 + 5*e_t^2 - 1:-y^5})

In [48]:
show((e_ns_3-e_ns_3_og).expand())

0

In [49]:
e_s_3=(de_dt_3PN_term.subs({e_t^2-1:-y}).subs({-e_t^2+1:y}).expand()-de_dt_3PN_term.subs({e_t^2-1:-y}).subs({-e_t^2+1:y}).expand().coefficient(SO,0)).expand()
e_s_3_og=de_dt.coefficient(Xi,6).subs({e_t^10 - 5*e_t^8 + 10*e_t^6 - 10*e_t^4 + 5*e_t^2 - 1:-y^5}).subs({e_t^2-1:-y}).subs({-e_t^2+1:y}).expand()

In [50]:
show((e_s_3-e_s_3_og).expand())

0

In [51]:
de_dt_final=de_dt_N_term+de_dt_1PN_term*Xi^2+de_dt_15PN_term*Xi^3+de_dt_2PN_term*Xi^4+de_dt_25PN_term*Xi^5+de_dt_3PN_term*Xi^6

**Newtonian**

In [52]:
dx_de_N_integrand=(dx_dt_N_term/de_dt_N_term)/x

In [53]:
dx_de_N=dx_de_N_integrand.integrate(e_t).subs({- 6/19*log(e_t^2):log(e_t^(-12/19))}).subs({-870/2299*log(121*e_t^2 + 304):log((121*e_t^2 + 304)^(-870/2299))})

In [54]:
dx_de_N_2=dx_de_N-dx_de_N.subs({e_t:e0})

In [55]:
x_N=(exp(dx_de_N_2).simplify_full())

In [56]:
show(x_N)

(121*e0^2 + 304)^(870/2299)*(e_t^2 - 1)*e0^(12/19)/((e0^2 - 1)*(121*e_t^2 + 304)^(870/2299)*e_t^(12/19))

In [57]:
e_factor=(e_t^(12/19)/e0^(12/19))


In [58]:
x_N_1=((x_N*e_factor).expand().taylor((e_t,0),(e0,0),6))*(x0/e_factor)

In [59]:
show((((x_N_1*x0)/(x0/e_factor)).expand()-xe_expr.coefficient(Xi,0)).expand())

0

**1 PN**

In [60]:
dx_de_1PN_integrand=(((dx_dt_N_term+dx_dt_1PN_term*Xi^2)/(de_dt_N_term+de_dt_1PN_term*Xi^2))/x)

For the step below you can also just do dx_de_1PN_integrand.taylor(Xi,0,2) and then do .coefficient(Xi,2).taylor(e_t,...))

In [61]:
dx_de_1PN_integrand_2=dx_de_1PN_integrand.taylor((Xi,0),(e_t,0),8).coefficient(Xi,2)

In [62]:
dx_de_1PN_integrand_3=dx_de_1PN_integrand_2.factor().subs({x:x_N_1}).taylor((e_t,0),(e0,0),6)

In [63]:
dx_de_1PN=(dx_de_1PN_integrand_3.integrate(e_t))*Xi^2

In [64]:
dx_de_1PN_2=dx_de_1PN-dx_de_1PN.subs({e_t:e0})

In [65]:
x_1PN=((exp(dx_de_1PN_2).simplify_full())*x_N).simplify_full()

In [66]:
x_1PN_1=(x_1PN.taylor((Xi,0),(e_t,0),(e0,0),10).coefficient(Xi,2))

In [67]:
x_1PN_final=((x_1PN_1*(e_factor)).expand().taylor((e_t,0),(e0,0),6))*(x0/e_factor)

In [68]:
show(((x_1PN_final/(1/e_factor)).expand()-xe_expr.coefficient(Xi,2)).expand())

0

**1.5 PN**

In [69]:
dx_de_15PN_integrand=(((dx_dt_N_term+dx_dt_1PN_term*Xi^2+dx_dt_15PN_term*Xi^3)/(de_dt_N_term+de_dt_1PN_term*Xi^2+de_dt_15PN_term*Xi^3))/x)

In [70]:
dx_de_15PN_integrand_2=dx_de_15PN_integrand.taylor((Xi,0),(e_t,0),9).coefficient(Xi,3)

In [71]:
dx_de_15PN_integrand_3=(dx_de_15PN_integrand_2.collect(x).subs({x:x_N_1})).taylor((e_t,0),(e0,0),6)

In [72]:
dx_de_15PN=(dx_de_15PN_integrand_3.integrate(e_t))*Xi^3

In [73]:
dx_de_15PN_2=dx_de_15PN-dx_de_15PN.subs({e_t:e0})

In [74]:
x_15PN=(((exp(dx_de_15PN_2).simplify_full())*x_1PN).simplify_full())

In [75]:
x_15PN_1=(x_15PN.taylor((Xi,0),(e_t,0),(e0,0),10).coefficient(Xi,3))

In [76]:
x_15PN_final=((x_15PN_1*(e_factor)).expand().taylor((e_t,0),(e0,0),6))*(x0/e_factor)

In [77]:
show(((x_15PN_final.subs({flagTail:1})*e_factor).expand()-xe_expr.coefficient(Xi,3)).expand())

0

**2 PN**

In [78]:
dx_de_2PN_integrand=(((dx_dt_N_term+dx_dt_1PN_term*Xi^2+dx_dt_15PN_term*Xi^3+dx_dt_2PN_term*Xi^4)/(de_dt_N_term+de_dt_1PN_term*Xi^2+de_dt_15PN_term*Xi^3+de_dt_2PN_term*Xi^4))/x)

In [79]:
dx_de_2PN_integrand_2=dx_de_2PN_integrand.taylor((Xi,0),(e_t,0),10).coefficient(Xi,4)

In [80]:
dx_de_2PN_integrand_3=dx_de_2PN_integrand_2.collect(x).subs({x:x_N_1}).taylor((e_t,0),(e0,0),6)

In [81]:
dx_de_2PN_integrand_4=(dx_de_1PN_integrand_2.collect(x).subs({x:x_N_1+Xi^2*x_1PN_final}).taylor((Xi,0),(e_t,0),(e0,0),10).coefficient(Xi,2))*Xi^4

In [84]:
dx_de_2PN=(dx_de_2PN_integrand_3.integrate(e_t)+dx_de_2PN_integrand_4.coefficient(Xi,4).expand().integrate(e_t))*Xi^4

In [85]:
dx_de_2PN_2=dx_de_2PN-dx_de_2PN.subs({e_t:e0})

In [86]:
x_2PN=(((exp(dx_de_2PN_2).simplify_full())*x_15PN).simplify_full())

In [138]:
x_2PN_1=(x_2PN.taylor((Xi,0),(e_t,0),(e0,0),12).coefficient(Xi,4))

In [139]:
x_2PN_final=((x_2PN_1*(e_factor)).expand().taylor((e_t,0),(e0,0),6))*(x0/e_factor)

PROOF THAT HENRY 2 PN CANNOT BE TRUSTED

In [140]:
show(((x_2PN_final.subs({flagTail:1})*e_factor).expand()-xe_expr.coefficient(Xi,4).subs({(e0^12)^(1/19):e0^(12/19)}).subs({(e0^88)^(1/19):e0^(88/19)})).expand())

-93941697725023/218588977152*e0^6*eta*x0^3 + 656053520979/1868281856*e0^(100/19)*e_t^(14/19)*eta*x0^3 + 5164560367101/24287664128*e0^4*e_t^2*eta*x0^3 - 2896831985721/24287664128*e0^(62/19)*e_t^(52/19)*eta*x0^3 - 2441614545/266897408*e0^2*e_t^4*eta*x0^3 - 175205335655/31226996736*e0^(24/19)*e_t^(90/19)*eta*x0^3 - 5525982219119/218588977152*e0^6*x0^3 + 38591383587/1868281856*e0^(100/19)*e_t^(14/19)*x0^3 + 303797668653/24287664128*e0^4*e_t^2*x0^3 - 170401881513/24287664128*e0^(62/19)*e_t^(52/19)*x0^3 - 143624385/266897408*e0^2*e_t^4*x0^3 - 10306196215/31226996736*e0^(24/19)*e_t^(90/19)*x0^3 - 1554186087/8409856*e0^4*eta*x0^3 + 26607261/161728*e0^(62/19)*e_t^(14/19)*eta*x0^3 + 8869087/161728*e0^2*e_t^2*eta*x0^3 - 290584009/8409856*e0^(24/19)*e_t^(52/19)*eta*x0^3 - 91422711/8409856*e0^4*x0^3 + 1565133/161728*e0^(62/19)*e_t^(14/19)*x0^3 + 521711/161728*e0^2*e_t^2*x0^3 - 17093177/8409856*e0^(24/19)*e_t^(52/19)*x0^3 - 2669/56*e0^2*eta*x0^3 + 2669/56*e0^(24/19)*e_t^(14/19)*eta*x0^3 - 157/56*e0^2*x0^3 + 157/56*e0^(24/19)*e_t^(14/19)*x0^3

In [143]:
filename = "x_e_files/x_2PN.txt"


with open(filename, "w") as f:
    f.write("# Expansion of x_2PN\n")
    f.write("# Plain ASCII output from Sage\n\n")
    f.write("x_2PN =\n")
    f.write(str(x_N_1+x_1PN_final*Xi^2+x_15PN_final*Xi^3+x_2PN_final*Xi^4))
    f.write("\n\n")